In [ ]:
import torch

In [ ]:
# temperature data in celcius
t_c = [0.5, 14.0, 15.0, 28.0, 11.0, 8.0, 3.0, -4.0, 6.0, 13.0, 21.0]

# temperature in unknown units
t_u = [35.7, 55.9, 58.2, 81.9, 56.3, 48.9, 33.9, 21.8, 48.4, 60.4, 68.4]

t_c = torch.tensor(t_c)
t_u = torch.tensor(t_u)

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(t_u, t_c)
plt.xlabel("Measurement")
plt.ylabel("Celcius")

In [ ]:
# linear model (for obviously linear data)
def model(t_u, w, b):
    ''' 
    Inputs
    ------
        t_u : PyTorch Tensor, model inputs
        w   : PyTorch Parameter, weight
        b   : PyTorch Parameter, bias
    
    '''
    return w * t_u + b

# squared error loss function
def loss_fn(t_p, t_c):
    ''' 
    Inputs
    ------
        t_p : PyTorch Tensor, predictions
        t_c : PyTorch Tensor, true values
    Outputs
    -------
        L : Float, loss function
    '''
    squared_diffs = (t_p - t_c)**2
    return squared_diffs.mean()

In [ ]:
# we want out weights and biases to be floats
# which implies zero dimensional tensors
# if we wrote, say, torch.ones(1) that would
# technically give us a 1 dimensional tensor
w = torch.ones(())
b = torch.zeros(())

w, b

In [ ]:
t_p = model(t_u, w, b)
t_p

In [ ]:
loss = loss_fn(t_p, t_c)
loss

In [ ]:
# numerical approximation to derivative
delta = 0.1
loss_rate_of_change_w = (
    loss_fn(model(t_u, w + delta, b),t_c) - loss_fn(model(t_u, w - delta, b),t_c)
) / (2.0 * delta)

loss_rate_of_change_w

In [ ]:
learning_rate = 1e-2

w = w - learning_rate * loss_rate_of_change_w
w

In [ ]:
loss_rate_of_change_b = (
    loss_fn(model(t_u, w, b+ delta),t_c) - loss_fn(model(t_u, w, b - delta),t_c)
) / (2.0 * delta)

loss_rate_of_change_b

In [ ]:
learning_rate = 1e-2

b = b - learning_rate * loss_rate_of_change_b
b

In [ ]:
def dloss_fun(t_p, t_c):
    dsq_diffs = 2 * (t_p - t_c) / t_p.size(0) # size to get the 1/N from the loss function
    return dsq_diffs

In [ ]:
def dmodel_dw(t_u, w, b):
    return t_u

def dmodel_db(t_u, w, b):
    return 1.0

In [ ]:
def grad_fn(t_u, t_c, t_p, w, b):
    # loss w.r.t. model predictions
    dloss_dtp = dloss_fun(t_p, t_c)
    
    # loss w.r.t. w
    dloss_dw = dloss_dtp * dmodel_dw(t_u, w, b) 

    # loss w.r.t. b
    dloss_db = dloss_dtp * dmodel_db(t_u, w, b)
    
    # all derivatives output a tensor (value for each data point)
    # so the derivatives w.r.t. the parameters are summed across 
    # all the data points
    # torch stack concatentates tensors along a new dimension
    # thus these zero dimensional tensors are made into a row vector
    return torch.stack([dloss_dw.sum(), dloss_db.sum()]) 

In [ ]:
def training_loop(n_epochs, learning_rate, params, t_u, t_c):
    # storage for later visual inspection
    params_data = torch.zeros(n_epochs+1,2)
    params_data[0] = params
    # fixed number of training steps for illustrative purposes
    for epoch in range(1,n_epochs + 1):
        # get latest parameter values
        w, b = params

        # make model predictions
        t_p = model(t_u, w, b)

        # calculate the loss
        loss = loss_fn(t_p, t_c)

        # calculate the gradient
        grad = grad_fn(t_u, t_c, t_p, w, b)
        
        # update the parmaters
        params = params - learning_rate * grad
        params_data[epoch] = params
        print(f"Epoch {epoch}, Loss = {loss}")
        print(f"\tParams: {params}")
        print(f"\tGrad: {grad}")
    
    return params_data


In [ ]:
training_loop(
    n_epochs = 100,
    learning_rate = 1e-2,
    params = torch.tensor([1.0,0.0]),
    t_u = t_u,
    t_c = t_c
)

In [ ]:
param_data = training_loop(
    n_epochs = 100,
    learning_rate = 1e-4,
    params = torch.tensor([1.0,0.0]),
    t_u = t_u,
    t_c = t_c
)

In [ ]:
plt.plot(param_data[:,0])
plt.xlabel('Iterations')
plt.ylabel('Weight')

In [ ]:
plt.plot(param_data[:,1])
plt.xlabel('Iterations')
plt.ylabel('Bias')

In [ ]:
# a coarse normalization scheme to ensure learning rate is applicable across all gradients
t_un = 0.1 * t_u

In [ ]:
param_data = training_loop(
    n_epochs = 100,
    learning_rate = 1e-2,
    params = torch.tensor([1.0,0.0]),
    t_u = t_un,
    t_c = t_c
)

In [ ]:
plt.plot(param_data[:,0])
plt.xlabel('Iterations')
plt.ylabel('Weight')

In [ ]:
plt.plot(param_data[:,1])
plt.xlabel('Iterations')
plt.ylabel('Bias')

In [ ]:
param_data = training_loop(
    n_epochs = 5000,
    learning_rate = 1e-2,
    params = torch.tensor([1.0,0.0]),
    t_u = t_un,
    t_c = t_c
)

In [ ]:
print(param_data[-1])

In [ ]:
plt.plot(param_data[:,0])
plt.xlabel('Iterations')
plt.ylabel('Weight')

In [ ]:
plt.plot(param_data[:,1])
plt.xlabel('Iterations')
plt.ylabel('Bias')

In [ ]:
# Lets visualize the results

t_p = model(t_un, *param_data[-1])
plt.scatter(t_u, t_p, label = "predictions")

plt.scatter(t_u, t_c, label = "actual")

import numpy as np
t_u_space = np.linspace(min(t_u), max(t_u), 100)
t_un_space = np.linspace(min(t_un), max(t_un), 100)
plt.plot(t_u_space, t_un_space*param_data[-1,0] + param_data[-1,1])

plt.legend()
plt.xlabel("Unknown Measurement Scale")
plt.ylabel("Celcius")